<a href="https://colab.research.google.com/github/adityacodes008/ADITYAKUMAR-.2026B03110246/blob/main/AdityaKumar_2026b03110246.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 3: Bike Rental Demand Analysis

**Student:** Aditya Kumar  
**Roll No:** 2026B03110246  
**Program:** Data Analytics

## 1. Data Ingestion & Initial Inspection
We load the bike rental dataset to inspect its structure, features, data types, and identify any missing values.

In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

# Input file should be in the same folder as this notebook
DATA_FILE = Path("03_bike_rental.csv")
OUTPUT_DIR = Path("bike_rental_notebook_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_FILE)

print("Dataset shape:", df.shape)
print("\nFirst five rows:")
display(df.head())

print("\nData types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isna().sum())


Dataset shape: (600, 9)

First five rows:


,Record_ID,DateTime,Season,Weather,Temperature_C,Humidity_Percent,Is_Weekend,Hour,Rental_Count
0,1,2025-01-01 00:00:00,Winter,Clear,42.0,83.0,Yes,0,36
1,2,2025-01-01 12:00:00,Autumn,Heavy Rain,23.1,65.0,Yes,12,34
2,3,2025-01-02 00:00:00,Summer,Cloudy,21.0,85.0,No,0,29
3,4,2025-01-02 12:00:00,Summer,Cloudy,31.1,29.0,No,12,33
4,5,2025-01-03 00:00:00,Autumn,Rain,25.7,81.0,Yes,0,34



Data types:


,0
Record_ID,int64
DateTime,object
Season,object
Weather,object
Temperature_C,float64
Humidity_Percent,float64
Is_Weekend,object
Hour,int64
Rental_Count,int64



Missing values:


,0
Record_ID,0
DateTime,0
Season,0
Weather,0
Temperature_C,0
Humidity_Percent,8
Is_Weekend,0
Hour,0
Rental_Count,0


## 2. Handling Missing Values
We handle missing humidity observations by imputing them with the dataset's median value to maintain data consistency.

In [5]:
humidity_missing_before = df["Humidity_Percent"].isna().sum()
humidity_median = df["Humidity_Percent"].median()

df["Humidity_Percent"] = df["Humidity_Percent"].fillna(humidity_median)

print("Missing humidity values before cleaning:", humidity_missing_before)
print("Median humidity used:", humidity_median)
print("Missing humidity values after cleaning:", df["Humidity_Percent"].isna().sum())


Missing humidity values before cleaning: 8
Median humidity used: 60.0
Missing humidity values after cleaning: 0


## 3. Date & Time Feature Engineering
We parse the timestamp strings into datetime objects and extract the Date, Month, and Hour features for finer analysis.

In [6]:
df["DateTime"] = pd.to_datetime(df["DateTime"], errors="coerce")
df["Date"] = df["DateTime"].dt.date
df["Month"] = df["DateTime"].dt.month
df["Hour_Extracted"] = df["DateTime"].dt.hour

display(df[["DateTime", "Date", "Month", "Hour_Extracted"]].head())
print("Recorded hour values:", sorted(df["Hour_Extracted"].dropna().unique()))


,DateTime,Date,Month,Hour_Extracted
0,2025-01-01 00:00:00,2025-01-01,1,0
1,2025-01-01 12:00:00,2025-01-01,1,12
2,2025-01-02 00:00:00,2025-01-02,1,0
3,2025-01-02 12:00:00,2025-01-02,1,12
4,2025-01-03 00:00:00,2025-01-03,1,0


Recorded hour values: [np.int32(0), np.int32(12)]


## 4. Aggregate Rental Statistics
Calculating overall descriptive statistics for bike rentals including total, average, median, minimum, and maximum counts.

In [7]:
total_rentals = df["Rental_Count"].sum()
average_rentals = df["Rental_Count"].mean()

print("Total rentals:", total_rentals)
print("Average rentals per record:", round(average_rentals, 2))
print("Median rentals per record:", df["Rental_Count"].median())
print("Minimum rentals:", df["Rental_Count"].min())
print("Maximum rentals:", df["Rental_Count"].max())


Total rentals: 21051
Average rentals per record: 35.08
Median rentals per record: 35.0
Minimum rentals: 18
Maximum rentals: 52


## 5. High-Demand Demand Records
Identifying the top 10 historical records with the highest registered bike rental demand.

In [8]:
top_10 = df.nlargest(10, "Rental_Count")
display(top_10)
top_10.to_csv(OUTPUT_DIR / "top_10_demand_records.csv", index=False)


,Record_ID,DateTime,Season,Weather,Temperature_C,Humidity_Percent,Is_Weekend,Hour,Rental_Count,Date,Month,Hour_Extracted
343,344,2025-06-21 12:00:00,Spring,Rain,23.2,41.0,Yes,12,52,2025-06-21,6,12
89,90,2025-02-14 12:00:00,Spring,Clear,12.8,49.0,No,12,50,2025-02-14,2,12
144,145,2025-03-14 00:00:00,Autumn,Clear,34.2,84.0,Yes,0,50,2025-03-14,3,0
453,454,2025-08-15 12:00:00,Winter,Rain,15.4,44.0,Yes,12,50,2025-08-15,8,12
92,93,2025-02-16 00:00:00,Summer,Cloudy,23.0,49.0,No,0,49,2025-02-16,2,0
141,142,2025-03-12 12:00:00,Autumn,Clear,28.7,77.0,Yes,12,48,2025-03-12,3,12
225,226,2025-04-23 12:00:00,Spring,Clear,24.0,82.0,No,12,48,2025-04-23,4,12
365,366,2025-07-02 12:00:00,Winter,Rain,22.3,91.0,No,12,48,2025-07-02,7,12
466,467,2025-08-22 00:00:00,Summer,Cloudy,21.4,91.0,No,0,48,2025-08-22,8,0
500,501,2025-09-08 00:00:00,Spring,Clear,32.2,65.0,Yes,0,48,2025-09-08,9,0


## 6. Demand Analysis by Season
Aggregating and analyzing average bike rental demands across different seasons.

In [9]:
season_summary = (
    df.groupby("Season")["Rental_Count"]
      .agg(Records="count", Total="sum", Average="mean")
      .sort_values("Average", ascending=False)
)
display(season_summary)

,Records,Total,Average
Season,,,
Spring,140,5033,35.950000
Autumn,167,5908,35.377246
Winter,159,5554,34.930818
Summer,134,4556,34.000000


## 7. Weekend vs. Weekday Patterns
Comparing rental behavior on weekends versus weekdays, with an additional calendar validation check.

In [10]:
weekend_summary = (
    df.groupby("Is_Weekend")["Rental_Count"]
      .agg(Records="count", Total="sum", Average="mean")
)
display(weekend_summary)

# Optional data-quality check: compare the supplied label with calendar weekdays
calendar_is_weekend = df["DateTime"].dt.dayofweek >= 5
provided_is_weekend = df["Is_Weekend"].astype(str).str.strip().str.lower().eq("yes")
agreement_pct = (calendar_is_weekend == provided_is_weekend).mean() * 100
print(f"Agreement between Is_Weekend label and calendar date: {agreement_pct:.1f}%")

,Records,Total,Average
Is_Weekend,,,
No,308,10692,34.714286
Yes,292,10359,35.476027


Agreement between Is_Weekend label and calendar date: 50.3%


## 8. Environmental Analysis: Weather Conditions
Examining how rental counts vary across different weather classifications.

In [11]:
weather_summary = (
    df.groupby("Weather")["Rental_Count"]
      .agg(Records="count", Total="sum", Average="mean")
      .sort_values("Average", ascending=False)
)
display(weather_summary)

,Records,Total,Average
Weather,,,
Clear,275,9808,35.665455
Rain,111,3902,35.153153
Heavy Rain,31,1065,34.354839
Cloudy,183,6276,34.295082


## 9. Hourly Demand Distribution
Analyzing how demand distributes across recorded hours of the day.

In [12]:
hourly_summary = (
    df.groupby("Hour_Extracted")["Rental_Count"]
      .agg(Records="count", Total="sum", Average="mean")
      .sort_index()
)
display(hourly_summary)


print("Note: The source dataset contains only the hour values shown above. Do not interpret this as a complete 24-hour demand profile.")


,Records,Total,Average
Hour_Extracted,,,
0,300,10451,34.836667
12,300,10600,35.333333


Note: The source dataset contains only the hour values shown above. Do not interpret this as a complete 24-hour demand profile.


## 10. Matrix Computations with NumPy
Validating global summary metrics using fast NumPy arrays.

In [13]:
average_temperature_numpy = np.mean(df["Temperature_C"].to_numpy())
average_rentals_numpy = np.mean(df["Rental_Count"].to_numpy())

print(f"NumPy average temperature: {average_temperature_numpy:.2f} °C")
print(f"NumPy average rental count: {average_rentals_numpy:.2f}")


NumPy average temperature: 24.06 °C
NumPy average rental count: 35.09


## 11. Statistical Outlier Detection (IQR Method)
Identifying unusually high rental count records using the Interquartile Range (IQR) technique.

In [14]:
q1 = df["Rental_Count"].quantile(0.25)
q3 = df["Rental_Count"].quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

high_rental_outliers = df[df["Rental_Count"] > upper_fence]

print(f"Q1: {q1:.2f}")
print(f"Q3: {q3:.2f}")
print(f"IQR: {iqr:.2f}")
print(f"Lower fence: {lower_fence:.2f}")
print(f"Upper fence: {upper_fence:.2f}")
print("Number of unusually high rental records:", len(high_rental_outliers))
display(high_rental_outliers)


Q1: 31.00
Q3: 39.00
IQR: 8.00
Lower fence: 19.00
Upper fence: 51.00
Number of unusually high rental records: 1


,Record_ID,DateTime,Season,Weather,Temperature_C,Humidity_Percent,Is_Weekend,Hour,Rental_Count,Date,Month,Hour_Extracted
343,344,2025-06-21 12:00:00,Spring,Rain,23.2,41.0,Yes,12,52,2025-06-21,6,12


## 12. Multi-dimensional Analysis: Season & Weather
Grouping demand by both Season and Weather simultaneously to locate sweet spots in operation.

In [15]:
season_weather_summary = (
    df.groupby(["Season", "Weather"])["Rental_Count"]
      .agg(Records="count", Average="mean", Total="sum")
      .sort_values("Average", ascending=False)
)
display(season_weather_summary)


Records    Average  Total
Season Weather                              
Spring Rain             22  36.727273    808
Winter Heavy Rain        6  36.666667    220
Spring Clear            62  36.467742   2261
Autumn Clear            72  36.166667   2604
Spring Heavy Rain        9  35.888889    323
Winter Clear            80  35.425000   2834
       Rain             33  35.393939   1168
Autumn Rain             30  35.266667   1058
Spring Cloudy           47  34.914894   1641
Autumn Cloudy           58  34.758621   2016
Summer Clear            61  34.573770   2109
       Cloudy           38  33.868421   1287
       Rain             26  33.384615    868
Winter Cloudy           40  33.300000   1332
Autumn Heavy Rain        7  32.857143    230
Summer Heavy Rain        9  32.444444    292

## 13. Ranked Demand Registry
Sorting and exporting the top 20 highest demand records.

In [16]:
sorted_by_demand = df.sort_values("Rental_Count", ascending=False)
display(sorted_by_demand.head(20))


,Record_ID,DateTime,Season,Weather,Temperature_C,Humidity_Percent,Is_Weekend,Hour,Rental_Count,Date,Month,Hour_Extracted
343,344,2025-06-21 12:00:00,Spring,Rain,23.2,41.0,Yes,12,52,2025-06-21,6,12
453,454,2025-08-15 12:00:00,Winter,Rain,15.4,44.0,Yes,12,50,2025-08-15,8,12
144,145,2025-03-14 00:00:00,Autumn,Clear,34.2,84.0,Yes,0,50,2025-03-14,3,0
89,90,2025-02-14 12:00:00,Spring,Clear,12.8,49.0,No,12,50,2025-02-14,2,12
92,93,2025-02-16 00:00:00,Summer,Cloudy,23.0,49.0,No,0,49,2025-02-16,2,0
365,366,2025-07-02 12:00:00,Winter,Rain,22.3,91.0,No,12,48,2025-07-02,7,12
500,501,2025-09-08 00:00:00,Spring,Clear,32.2,65.0,Yes,0,48,2025-09-08,9,0
225,226,2025-04-23 12:00:00,Spring,Clear,24.0,82.0,No,12,48,2025-04-23,4,12
141,142,2025-03-12 12:00:00,Autumn,Clear,28.7,77.0,Yes,12,48,2025-03-12,3,12
466,467,2025-08-22 00:00:00,Summer,Cloudy,21.4,91.0,No,0,48,2025-08-22,8,0


## 14. Correlation Analysis: Temperature vs. Demand
Calculating the Pearson correlation coefficient between temperature and demand to evaluate physical dependencies.

In [17]:
temperature_rental_correlation = df["Temperature_C"].corr(df["Rental_Count"])
print(f"Pearson correlation calculation: {temperature_rental_correlation:.4f}")
print("Interpretation: A value close to zero indicates little or no linear association in this sample. Correlation does not prove causation.")

Pearson correlation calculation: -0.0039
Interpretation: A value close to zero indicates little or no linear association in this sample. Correlation does not prove causation.


## 15. Five operational recommendations
Task 48.

1. **Seasonal fleet planning:** Use seasonal demand averages to guide gradual fleet allocation, while validating the pattern with more historical data.
2. **Weather-aware operations:** Monitor weather forecasts and adjust redistribution plans based on observed demand under different weather categories.
3. **Validate weekend labels:** Check the `Is_Weekend` field against calendar dates before using it for staffing or weekend fleet planning.
4. **Collect more detailed timestamps:** The dataset only contains a limited set of recorded hours. Collect hourly or 15-minute observations to identify actual demand peaks.
5. **Use multiple forecasting features:** Do not rely on temperature alone. Include validated time, season, weather, location and historical demand features in any future forecasting model.


## 15. Operational Recommendations & Next Steps

In [18]:
# Save the cleaned dataset and all requested summary tables
df.to_csv(OUTPUT_DIR / "bike_rental_cleaned.csv", index=False)
season_summary.to_csv(OUTPUT_DIR / "season_summary.csv")
weekend_summary.to_csv(OUTPUT_DIR / "weekday_weekend_summary.csv")
weather_summary.to_csv(OUTPUT_DIR / "weather_summary.csv")
hourly_summary.to_csv(OUTPUT_DIR / "hourly_summary.csv")
season_weather_summary.to_csv(OUTPUT_DIR / "season_weather_summary.csv")

print("All output files saved in:", OUTPUT_DIR.resolve())


All output files saved in: /content/bike_rental_notebook_outputs


## 16. Consolidated Business KPI Summary Table

In [19]:
# Calculate necessary values inline if they are not already in memory
tmp_avg_temp = average_temperature_numpy if 'average_temperature_numpy' in locals() else np.mean(df["Temperature_C"].to_numpy())
tmp_avg_rent_np = average_rentals_numpy if 'average_rentals_numpy' in locals() else np.mean(df["Rental_Count"].to_numpy())

if 'high_rental_outliers' in locals():
    tmp_outlier_count = len(high_rental_outliers)
else:
    q1 = df["Rental_Count"].quantile(0.25)
    q3 = df["Rental_Count"].quantile(0.75)
    iqr = q3 - q1
    upper_fence = q3 + 1.5 * iqr
    tmp_outlier_count = len(df[df["Rental_Count"] > upper_fence])

summary = pd.DataFrame({
    "Metric": [
        "Number of records",
        "Total rentals",
        "Average rentals per record",
        "Missing humidity values before cleaning",
        "Median humidity used for imputation",
        "Average temperature (NumPy)",
        "Average rentals (NumPy)",
        "High rental-count outliers (IQR)",
        "Temperature/rental correlation",
        "Weekend-label/calendar agreement (%)"
    ],
    "Value": [
        len(df),
        total_rentals,
        round(average_rentals, 2),
        int(humidity_missing_before),
        humidity_median,
        round(tmp_avg_temp, 2),
        round(tmp_avg_rent_np, 2),
        tmp_outlier_count,
        round(temperature_rental_correlation, 4),
        round(agreement_pct, 1)
    ]
})
display(summary)

,Metric,Value
0,Number of records,600.0000
1,Total rentals,21051.0000
2,Average rentals per record,35.0800
3,Missing humidity values before cleaning,8.0000
4,Median humidity used for imputation,60.0000
5,Average temperature (NumPy),24.0600
6,Average rentals (NumPy),35.0800
7,High rental-count outliers (IQR),1.0000
8,Temperature/rental correlation,-0.0039
9,Weekend-label/calendar agreement (%),50.3000
